In [ ]:
from pathlib import Path
from metasmith.python_api import Agent, Source, SshSource, Std, DataInstanceLibrary, WorkflowTask
from local.constants import WORKSPACE_ROOT

dtypes, containers, transforms = Std()

# agent_home = Source.FromLocal(Path("./cache/local_home").resolve())
agent_home = SshSource("fir", Path("/scratch/phyberos/metasmith")).AsSource()
smith = Agent(
    home = agent_home,
)
# smith.Deploy()

In [ ]:
inputs = DataInstanceLibrary("./cache/reads.xgdb")
inputs.Add(
    items=[
        (WORKSPACE_ROOT/"main/local_mock/cache/flye_in/lr_ss10.fastq", "lr.fq", "std::hifi_reads"),
        (WORKSPACE_ROOT/"main/local_mock/cache/asm.xgdb/scadc.fna", "scadc.fna", "std::assembly"),
    ]
)
inputs.Save()
for p, n, e in inputs.Iterate():
    print(n, p, e, e.parents)

In [ ]:
# dtypes.types

In [ ]:
_tasks = [
    smith.GenerateWorkflow(
        given      = [containers, inputs],
        transforms = [transforms],
        targets    = [dtypes[t]]
    )
    for t in ["per_contig_coverage"]
]

task = WorkflowTask.Merge((_tasks))
for step in [s for p in task.plans for s in p.steps]:
    print(step.order, step.transform.name)
# task.RenderDAG("./cache/dag")

In [ ]:
smith.StageWorkflow(task, on_exist="clear")

In [ ]:
smith.RunWorkflow(task)

In [ ]:
smith.CheckWorkflow(task)